# Correlation Tutorial 2: Ecosystem Interoperability with ASE & Pymatgen

Correlation integrates seamlessly into Python materials science pipelines.

This tutorial covers:
1. Converting between **ASE `Atoms`** and **Correlation `Cell`**.
2. Converting between **Pymatgen `Structure`** and **Correlation `Cell`**.
3. Converting multi-frame **trajectories**.
4. Using convenient monkey-patched conversion methods.


## 1. Prerequisites and Setup


In [ ]:
import correlation
import numpy as np

# Check optional dependencies
try:
    import ase
    from ase.build import bulk, molecule
    has_ase = True
    print(f"ASE version: {ase.__version__}")
except ImportError:
    has_ase = False
    print("ASE not installed. Install via: pip install ase")

try:
    import pymatgen.core as pmg
    has_pmg = True
    print(f"Pymatgen version: {pmg.__version__}")
except ImportError:
    has_pmg = False
    print("Pymatgen not installed. Install via: pip install pymatgen")


## 2. Interoperability with ASE

Correlation provides bidirectional conversions via `correlation.from_ase()` and `correlation.to_ase()`.


In [ ]:
if has_ase:
    # 1. Build a supercell using ASE
    cu_fcc = bulk('Cu', 'fcc', a=3.61, cubic=True) * (2, 2, 2)
    print(f"ASE Atoms: {cu_fcc.get_chemical_formula()} with {len(cu_fcc)} atoms")
    print(f"Cell matrix:\n{cu_fcc.get_cell()[:]}")

    # 2. Convert to Correlation Cell
    corr_cell = correlation.from_ase(cu_fcc)
    print(f"Correlation Cell: {corr_cell.atom_count()} atoms, volume: {corr_cell.volume:.2f} Å³")

    # 3. Roundtrip back to ASE Atoms
    cu_roundtrip = correlation.to_ase(corr_cell)
    np.testing.assert_allclose(cu_fcc.get_positions(), cu_roundtrip.get_positions(), atol=1e-5)
    np.testing.assert_allclose(cu_fcc.get_cell()[:], cu_roundtrip.get_cell()[:], atol=1e-5)
    print("ASE roundtrip validation: 100% SUCCESS")


## 3. Interoperability with Pymatgen

Similarly, `correlation.from_pymatgen()` and `correlation.to_pymatgen()` provide full fidelity conversions.


In [ ]:
if has_pmg:
    # 1. Create a Pymatgen Structure (NaCl rocksalt)
    lattice = pmg.Lattice.cubic(5.64)
    structure = pmg.Structure(
        lattice,
        ["Na", "Cl"],
        [[0.0, 0.0, 0.0], [0.5, 0.5, 0.5]]
    )
    print(f"Pymatgen Structure: {structure.formula}, volume: {structure.volume:.2f} Å³")

    # 2. Convert to Correlation Cell
    corr_cell = correlation.from_pymatgen(structure)
    print(f"Correlation Cell: {corr_cell.atom_count()} atoms, volume: {corr_cell.volume:.2f} Å³")

    # 3. Roundtrip back to Pymatgen Structure
    pmg_roundtrip = correlation.to_pymatgen(corr_cell)
    np.testing.assert_allclose(structure.lattice.matrix, pmg_roundtrip.lattice.matrix, atol=1e-5)
    print("Pymatgen roundtrip validation: 100% SUCCESS")


## 4. Monkey-Patched Convenience Methods

Correlation injects `.to_correlation()` into `ase.Atoms` and `pymatgen.core.Structure`, and `.to_ase()` / `.to_pymatgen()` into `correlation.Cell`.


In [ ]:
if has_ase:
    water = molecule('H2O')
    # Direct method call on ASE Atoms!
    corr_water = water.to_correlation()
    print(f"Converted water: {corr_water.atom_count()} atoms")

    # Direct method call on Correlation Cell!
    ase_water = corr_water.to_ase()
    print(f"Back to ASE: {ase_water.get_chemical_formula()}")


## 5. End-to-End Workflow: Perturbation and RDF Calculation

We can perturb an atomic structure using ASE, compute its pair correlation using Correlation's high-performance C++ engine, and compare it with the perfect crystal.


In [ ]:
if has_ase:
    import matplotlib.pyplot as plt

    # Generate perfect and thermally perturbed crystals
    perfect_cu = bulk('Cu', 'fcc', a=3.61, cubic=True) * (4, 4, 4)
    perturbed_cu = perfect_cu.copy()
    perturbed_cu.rattle(stdev=0.08, seed=42)  # Thermal jitter

    # Compute RDF for both using Correlation
    df_perf = correlation.DistributionFunctions(perfect_cu.to_correlation())
    df_perf.calculate_rdf(correlation.RDFParams(r_max=8.0, r_bin_width=0.02))

    df_pert = correlation.DistributionFunctions(perturbed_cu.to_correlation())
    df_pert.calculate_rdf(correlation.RDFParams(r_max=8.0, r_bin_width=0.02))

    # Plot comparison
    r = df_perf.get_histogram("g_r").bins
    g_perf = df_perf.get_histogram("g_r").partials["Total"]
    g_pert = df_pert.get_histogram("g_r").partials["Total"]

    plt.figure(figsize=(8, 4), dpi=120)
    plt.plot(r, g_perf, label="Ideal FCC Cu (0 K)", lw=1.5, alpha=0.7)
    plt.plot(r, g_pert, label="Perturbed Cu (Thermal jitter)", lw=2.0, color="#D55E00")
    plt.xlabel("Radius r (Å)")
    plt.ylabel("g(r)")
    plt.title("Comparison of Ideal vs. Thermally Jittered FCC Copper")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


## Conclusion
You can now freely transition between ASE, Pymatgen, and Correlation without serializing files to disk.

Next tutorial: `03_trajectory_and_dynamics.ipynb` explores time-series correlation functions (MSD, VACF, and VDOS).
